# Notebook 03 — EDA, Visualisasi & Insight

**Tugas 1 · Analisis Big Data · Semester Ganjil 2026**

Notebook ini dipakai pada **Milestone 3 & Final Submission**:

1. Analisis pola temporal (time series) dengan Polars.
2. Minimal **6 visualisasi interaktif** Plotly/Altair.
3. Menuliskan **5+ insight** analitik.

Simpan setiap figure ke `output/figures/`.


## 1. Muat Data Bersih


In [ ]:
import plotly.express as px
import polars as pl

from src.data_loading import resolve_data_path

df = pl.read_parquet(resolve_data_path("processed/dataset_bersih.parquet"))
print(f"{df.height:,} baris x {df.width} kolom")


## 2. Analisis Temporal

Agregasi per periode/window. Sesuaikan kolom tanggal & ukuran window.


In [ ]:
# TODO: sesuaikan kolom tanggal dan interval window
df_temporal = (
    df.sort("tanggal")
    .group_by_dynamic("tanggal", every="1mo")
    .agg(
        pl.len().alias("jumlah"),
        pl.col("harga_clean").sum().alias("total"),
    )
)
df_temporal


## 3. Visualisasi Interaktif

Minimal 6 chart berbeda. Contoh di bawah bisa langsung dicopy & disesuaikan.


### 3.1 Bar — Top kategori/kota


In [ ]:
# TODO: sesuaikan kolom
agg = (df.group_by("kategori").agg(pl.len().alias("jumlah")).sort("jumlah", descending=True))

fig = px.bar(agg.to_pandas(), x="kategori", y="jumlah", title="Jumlah per Kategori")
fig.write_html("output/figures/01_bar.html")
fig.show()


### 3.2 Line — Tren temporal


In [ ]:
fig = px.line(
    df_temporal.to_pandas(),
    x="tanggal",
    y="total",
    title="Tren Total per Bulan",
)
fig.write_html("output/figures/02_line.html")
fig.show()


### 3.3 Scatter — Hubungan dua variabel


In [ ]:
sample = df.sample(n=min(10_000, df.height), seed=42)
fig = px.scatter(
    sample.to_pandas(),
    x="harga_clean",
    y="kuantitas",
    color="kategori",
    opacity=0.5,
    title="Harga vs Kuantitas",
)
fig.write_html("output/figures/03_scatter.html")
fig.show()


### 3.4 Histogram — Distribusi


In [ ]:
fig = px.histogram(df.to_pandas(), x="harga_clean", nbins=50, title="Distribusi Harga")
fig.write_html("output/figures/04_histogram.html")
fig.show()


### 3.5 Box — Sebaran per kategori (deteksi outlier)


In [ ]:
fig = px.box(df.to_pandas(), x="kategori", y="harga_clean", title="Sebaran Harga per Kategori")
fig.write_html("output/figures/05_box.html")
fig.show()


### 3.6 Chart ke-6 (pilih: peta choropleth / heatmap / pie / sunburst)


In [ ]:
# TODO: pilih satu jenis visualisasi tambahan
fig = px.pie(df.to_pandas(), names="kategori", title="Proporsi Kategori")
fig.write_html("output/figures/06_pie.html")
fig.show()


### Alternatif: Altair


In [ ]:
import altair as alt

chart = (
    alt.Chart(df.group_by("kategori").agg(pl.len().alias("jumlah")).to_pandas())
    .mark_bar()
    .encode(x="kategori", y="jumlah")
)
chart.save("output/figures/07_altair.html")
chart


## 4. Insight Utama (minimal 5)

> Tuliskan insight berbasis data, bukan deskripsi umum. Sertakan angka pendukung.
>
> 1. ...
> 2. ...
> 3. ...
> 4. ...
> 5. ...

---

**AI Disclosure:** catat penggunaan AI asisten untuk notebook ini di `README.md`.
